In [1]:
import numpy as np

encoded_sequences = np.load(
    "../encoded_sequences_len40_v2.npy"
)

print(encoded_sequences.shape)

(7405, 42)


In [2]:
import torch

X = torch.tensor(
    encoded_sequences,
    dtype=torch.long
)

In [3]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=False
)

In [4]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [5]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(
    nn.Module
):

    def __init__(
        self,
        d_model,
        max_len=42
    ):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            )
            *
            (
                -torch.log(
                    torch.tensor(
                        10000.0
                    )
                )
                /
                d_model
            )
        )

        pe[:,0::2] = torch.sin(
            position *
            div_term
        )

        pe[:,1::2] = torch.cos(
            position *
            div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(
        self,
        x
    ):

        return (
            x
            +
            self.pe[:,:x.size(1)]
        )

In [6]:
class TransformerVAE(
    nn.Module
):

    def __init__(
        self,
        vocab_size=23,
        embed_dim=128,
        latent_dim=64,
        max_len=42
    ):

        super().__init__()

        self.max_len = max_len

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.pos_encoder = (
            PositionalEncoding(
                embed_dim,
                max_len
            )
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=8,
                batch_first=True
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=3
            )
        )

        self.fc_mu = nn.Linear(
            embed_dim,
            latent_dim
        )

        self.fc_logvar = nn.Linear(
            embed_dim,
            latent_dim
        )

        self.latent_to_embed = (
            nn.Linear(
                latent_dim,
                embed_dim
            )
        )

        decoder_layer = (
            nn.TransformerDecoderLayer(
                d_model=embed_dim,
                nhead=8,
                batch_first=True
            )
        )

        self.decoder = (
            nn.TransformerDecoder(
                decoder_layer,
                num_layers=3
            )
        )

        self.output_layer = nn.Linear(
            embed_dim,
            vocab_size
        )

    def encode(
        self,
        x
    ):

        x = self.embedding(x)

        x = self.pos_encoder(x)

        enc = self.encoder(x)

        pooled = enc.mean(dim=1)

        mu = self.fc_mu(
            pooled
        )

        logvar = self.fc_logvar(
            pooled
        )

        return mu, logvar
        
    def reparameterize(
        self,
        mu,
        logvar
    ):

        std = torch.exp(
            0.5 * logvar
        )

        eps = torch.randn_like(
            std
        )

        return (
            mu
            +
            eps * std
        )

    def decode(
        self,
        z,
        decoder_input
    ):

        tgt = self.embedding(
            decoder_input
        )

        tgt = self.pos_encoder(
            tgt
        )

        memory = (
            self.latent_to_embed(
                z
            )
            .unsqueeze(1)
        )

        out = self.decoder(
            tgt,
            memory
        )

        logits = self.output_layer(
            out
        )

        return logits

    def forward(
        self,
        x
    ):

        mu, logvar = self.encode(
            x
        )

        z = self.reparameterize(
            mu,
            logvar
        )

        decoder_input = x[:,:-1]

        logits = self.decode(
            z,
            decoder_input
        )

        return (
            logits,
            mu,
            logvar
        )

In [7]:
vae = TransformerVAE(
    vocab_size=23,
    max_len=42
).to(device)

In [9]:
vae.eval()

latent_list = []

with torch.no_grad():

    for batch in loader:

        x = batch[0].to(device)

        mu, logvar = vae.encode(x)

        latent_list.append(
            mu.cpu()
        )

latent_vectors = torch.cat(
    latent_list,
    dim=0
).numpy()

print(
    latent_vectors.shape
)

(7405, 64)


In [10]:
print(latent_vectors.mean())
print(latent_vectors.std())
print(latent_vectors.min())
print(latent_vectors.max())

-0.072846614
0.42651695
-1.485657
1.1084514


In [11]:
import numpy as np

np.save(
    "latent_vectors_len40_v2.npy",
    latent_vectors
)

print("Saved")

Saved


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

latent_scaled = scaler.fit_transform(
    latent_vectors
)

print(latent_scaled.mean())
print(latent_scaled.std())

-5.1515153e-10
1.0


In [13]:
import joblib
joblib.dump(
    scaler,
    "latent_scaler_len40_v2.pkl"
)

np.save(
    "latent_scaled_len40_v2.npy",
    latent_scaled
)

In [14]:
print(latent_scaled.shape)

print(latent_scaled.mean())
print(latent_scaled.std())

print(latent_scaled.min())
print(latent_scaled.max())

(7405, 64)
-5.1515153e-10
1.0
-8.952018
8.702067
